# NB02: Seoul Building Clip and Height Features

This notebook clips the national facility-building shapefile to the Seoul administrative boundary created in NB01, then adds building height and floor-derived fields used by later constraint scoring and dashboard preparation.

Inputs:
- `processed/seoul_boundary.gpkg`, layer `dong_5179`
- `00_data/F_FAC_BUILDING_서울/F_FAC_BUILDING_11_202604.shp`

Outputs:
- `processed/seoul_buildings.gpkg`, layer `buildings_5179`
- `processed/seoul_buildings.gpkg`, layer `buildings_4326`

Note: the layer name `buildings_5179` is kept for compatibility with later notebooks, but the source CRS is checked and preserved from the building shapefile.

## 1. Imports and Project Paths

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio


def find_project_root() -> Path:
    """Return the skybase-seoul-v3 project root when run from this notebook."""
    cwd = Path.cwd().resolve()
    if cwd.name == "01_preprocessing" and (cwd.parent / "00_data").exists():
        return cwd.parent
    if (cwd / "00_data").exists() and cwd.name == "skybase-seoul-v3":
        return cwd
    for parent in [cwd, *cwd.parents]:
        candidate = parent / "skybase-seoul-v3"
        if (candidate / "00_data").exists():
            return candidate
    raise FileNotFoundError("Could not locate skybase-seoul-v3/00_data from the current working directory.")


BASE = find_project_root()
RAW = BASE / "00_data"
OUT = BASE / "processed"
OUT.mkdir(parents=True, exist_ok=True)

print(f"BASE: {BASE}")
print(f"RAW : {RAW}")
print(f"OUT : {OUT}")

BASE: C:\Users\jimin\Desktop\1_BITAmin_16기\skybase-seoul\skybase-seoul-v3
RAW : C:\Users\jimin\Desktop\1_BITAmin_16기\skybase-seoul\skybase-seoul-v3\00_data
OUT : C:\Users\jimin\Desktop\1_BITAmin_16기\skybase-seoul\skybase-seoul-v3\processed


## 2. Load Seoul Boundary and Building Source

In [2]:
boundary_path = OUT / "seoul_boundary.gpkg"
if not boundary_path.exists():
    raise FileNotFoundError(f"Run NB01 first: {boundary_path}")

seoul = gpd.read_file(boundary_path, layer="dong_5179")
if seoul.empty:
    raise ValueError("Seoul boundary layer is empty")

building_files = sorted(RAW.rglob("F_FAC_BUILDING_11_*.shp"))
if not building_files:
    raise FileNotFoundError("Could not find F_FAC_BUILDING_11_*.shp under 00_data")
building_path = building_files[0]
building_info = pyogrio.read_info(building_path)
building_crs = building_info["crs"]

print(f"Boundary dongs: {len(seoul):,}")
print(f"Boundary CRS  : {seoul.crs}")
print(f"Building file : {building_path}")
print(f"Building rows : {building_info['features']:,}")
print(f"Building CRS  : {building_crs}")

Boundary dongs: 426
Boundary CRS  : EPSG:5186
Building file : C:\Users\jimin\Desktop\1_BITAmin_16기\skybase-seoul\skybase-seoul-v3\00_data\F_FAC_BUILDING_서울\F_FAC_BUILDING_11_202604.shp
Building rows : 696,740
Building CRS  : EPSG:5186


## 3. Read Buildings with Seoul Bounding Box

In [3]:
seoul_for_building_crs = seoul.to_crs(building_crs) if str(seoul.crs) != str(building_crs) else seoul.copy()
bounds = seoul_for_building_crs.total_bounds
bbox_buffer_m = 500
bbox = (
    bounds[0] - bbox_buffer_m,
    bounds[1] - bbox_buffer_m,
    bounds[2] + bbox_buffer_m,
    bounds[3] + bbox_buffer_m,
)

print(f"Seoul bounds in building CRS: {bounds}")
print(f"Building read bbox (+{bbox_buffer_m} m): {bbox}")

buildings_bbox = gpd.read_file(building_path, bbox=bbox)
print(f"Buildings read inside bbox: {len(buildings_bbox):,}")
print(f"Columns: {buildings_bbox.columns.tolist()}")

Seoul bounds in building CRS: [179189.76163945 536547.40661053 216242.2727258  566863.54149205]
Building read bbox (+500 m): (np.float64(178689.76163944902), np.float64(536047.406610533), np.float64(216742.2727258006), np.float64(567363.5414920457))
Buildings read inside bbox: 696,737
Columns: ['UFID', 'BLD_NM', 'DONG_NM', 'GRND_FLR', 'UGRND_FLR', 'PNU', 'ARCHAREA', 'TOTALAREA', 'PLATAREA', 'HEIGHT', 'STRCT_CD', 'USABILITY', 'BC_RAT', 'VL_RAT', 'BLDRGST_PK', 'USEAPR_DAY', 'REGIST_DAY', 'GB_CD', 'VIOL_BD_YN', 'GEOIDN', 'BLDG_PNU', 'BLDG_PNU_Y', 'BLD_UNLICE', 'BD_MGT_SN', 'SGG_OID', 'COL_ADM_SE', 'geometry']


## 4. Clip to Seoul Boundary

In [4]:
city_boundary = seoul_for_building_crs[["geometry"]].dissolve()
buildings_seoul = gpd.sjoin(
    buildings_bbox,
    city_boundary[["geometry"]],
    predicate="intersects",
    how="inner",
).drop(columns="index_right")

buildings_seoul = buildings_seoul.reset_index(drop=True)
print(f"Buildings inside Seoul boundary: {len(buildings_seoul):,}")
print(f"Removed by precise boundary clip: {len(buildings_bbox) - len(buildings_seoul):,}")
display(buildings_seoul.drop(columns="geometry").head())

Buildings inside Seoul boundary: 696,712
Removed by precise boundary clip: 25


,UFID,BLD_NM,DONG_NM,GRND_FLR,UGRND_FLR,PNU,ARCHAREA,TOTALAREA,PLATAREA,HEIGHT,...,REGIST_DAY,GB_CD,VIOL_BD_YN,GEOIDN,BLDG_PNU,BLDG_PNU_Y,BLD_UNLICE,BD_MGT_SN,SGG_OID,COL_ADM_SE
0,1991201839054527769900000000,None,None,3.0,0.0,1111017500107040000,57.0,203,100.0,9.0,...,20170530,0,0,B00100000000T30Z9,None,None,None,1111017500107040000001651,32,11110
1,1967201343844527773800000000,None,None,3.0,0.0,1111017500100560024,0.0,114,0.0,0.0,...,20180619,0,0,B00100000000T311B,None,None,None,1111017500100560024002963,34,11110
2,1962201375194527742000000000,None,None,2.0,0.0,1111017500100560054,0.0,135,0.0,0.0,...,20170224,0,0,B00100000000T313D,None,None,None,1111017500100560054002866,36,11110
3,1979201388484527710100000000,None,None,2.0,1.0,1111017500100560053,0.0,230,0.0,0.0,...,20181106,0,0,B00100000000T314E,None,None,None,1111017500100560053002771,37,11110
4,0000200468294527764000000000,None,None,0.0,0.0,1111016500100280011,0.0,0,0.0,0.0,...,None,None,None,B00100000000T315F,None,None,None,1111016500100280010009883,38,11110


## 5. Add Height and Floor Features

In [5]:
def numeric_col(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype="float64")
    return pd.to_numeric(df[column], errors="coerce")


floors = numeric_col(buildings_seoul, "GRND_FLR")
underground = numeric_col(buildings_seoul, "UGRND_FLR")
height_raw = numeric_col(buildings_seoul, "HEIGHT")
arch_area = numeric_col(buildings_seoul, "ARCHAREA")
total_area = numeric_col(buildings_seoul, "TOTALAREA")
plot_area = numeric_col(buildings_seoul, "PLATAREA")

height_from_raw = height_raw.where(height_raw.between(2, 300))
height_from_floors = (floors.where(floors >= 1) * 3).clip(upper=300)
height_m = height_from_raw.fillna(height_from_floors).fillna(0)

buildings_seoul["floors"] = floors.fillna(0)
buildings_seoul["underground_floors"] = underground.fillna(0)
buildings_seoul["height_m"] = height_m
buildings_seoul["arch_area"] = arch_area.fillna(0)
buildings_seoul["total_area"] = total_area.fillna(0)
buildings_seoul["plot_area"] = plot_area.fillna(0)

with np.errstate(divide="ignore", invalid="ignore"):
    density = buildings_seoul["total_area"] / buildings_seoul["plot_area"]
buildings_seoul["building_density"] = density.replace([np.inf, -np.inf], np.nan).fillna(0)
buildings_seoul["vertical_complexity"] = buildings_seoul["height_m"] * buildings_seoul["building_density"]
buildings_seoul["is_highrise"] = buildings_seoul["height_m"] > 30

summary = buildings_seoul[["floors", "height_m", "building_density", "vertical_complexity"]].describe().round(2)
print(f"High-rise buildings (>30 m): {buildings_seoul['is_highrise'].sum():,}")
display(summary)

High-rise buildings (>30 m): 19,513


,floors,height_m,building_density,vertical_complexity
count,696712.00,696712.00,696712.00,696712.00
mean,2.64,8.13,1.03,15.83
std,3.02,9.29,14.42,434.80
min,0.00,0.00,-0.00,-0.02
25%,1.00,3.00,0.00,0.00
50%,2.00,6.00,0.00,0.00
75%,4.00,11.00,1.92,19.37
max,69.00,284.00,9171.06,311816.02


## 6. Save Projected and Web-Mapping Layers

In [6]:
building_output = OUT / "seoul_buildings.gpkg"
if building_output.exists():
    building_output.unlink()

buildings_seoul.to_file(building_output, layer="buildings_5179", driver="GPKG")
buildings_4326 = buildings_seoul.to_crs(epsg=4326)
buildings_4326.to_file(building_output, layer="buildings_4326", driver="GPKG")

print(f"Saved: {building_output}")
print(f"  buildings_5179: {len(buildings_seoul):,} rows, CRS={buildings_seoul.crs}")
print(f"  buildings_4326: {len(buildings_4326):,} rows, CRS={buildings_4326.crs}")

Saved: C:\Users\jimin\Desktop\1_BITAmin_16기\skybase-seoul\skybase-seoul-v3\processed\seoul_buildings.gpkg
  buildings_5179: 696,712 rows, CRS=EPSG:5186
  buildings_4326: 696,712 rows, CRS=EPSG:4326


## 7. Validation

In [7]:
layers = {name for name, _ in pyogrio.list_layers(building_output)}
expected_layers = {"buildings_5179", "buildings_4326"}
missing_layers = expected_layers - layers
if missing_layers:
    raise AssertionError(f"Missing GeoPackage layers: {sorted(missing_layers)}")

check_5179 = gpd.read_file(building_output, layer="buildings_5179")
check_4326 = gpd.read_file(building_output, layer="buildings_4326")
required_cols = {
    "UFID", "GRND_FLR", "UGRND_FLR", "HEIGHT",
    "floors", "underground_floors", "height_m",
    "arch_area", "total_area", "plot_area",
    "building_density", "vertical_complexity", "is_highrise",
    "geometry",
}
missing_cols = required_cols - set(check_5179.columns)
if missing_cols:
    raise AssertionError(f"Missing required columns: {sorted(missing_cols)}")

assert len(check_5179) == len(check_4326), "Layer row counts differ"
assert len(check_5179) > 0, "No Seoul buildings were saved"
assert check_4326.crs.to_epsg() == 4326, "buildings_4326 must be EPSG:4326"
assert check_5179["height_m"].between(0, 300).all(), "height_m outside expected 0-300 m range"

print("Validation passed")
print(f"  layers          : {sorted(layers)}")
print(f"  building rows   : {len(check_5179):,}")
print(f"  projected CRS   : {check_5179.crs}")
print(f"  web CRS         : {check_4326.crs}")
print(f"  high-rise count : {check_5179['is_highrise'].sum():,}")
print(f"  max height_m    : {check_5179['height_m'].max():.1f}")

Validation passed
  layers          : ['buildings_4326', 'buildings_5179']
  building rows   : 696,712
  projected CRS   : EPSG:5186
  web CRS         : EPSG:4326
  high-rise count : 19,513
  max height_m    : 284.0
